In [1]:
import argparse
import os
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from model import DocumentReconstructionModel
from uv_dewarp import dewarp_with_uv
from dataset_loader import get_dataloaders
import torch.nn as nn
from pytorch_msssim import ssim

/Users/michaelbehrens/micromamba/envs/cs135_25f_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configuration
DATA_DIR = 'renders/synthetic_data_pitch_sweep'
BATCH_SIZE = 8
IMG_SIZE = (256, 256)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# load model
model = DocumentReconstructionModel().to(device)
state = torch.load('models/best_model.pth', map_location=device)
if isinstance(state, dict) and 'model_state' in state:
    state = state['model_state']
model.load_state_dict(state)
model.eval()
print(f"Loaded weights from models/best_model.pth")

train_loader, val_loader = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    use_depth=False,  # TODO: Set to True if you want to use depth information
    use_uv=True,     # TODO: Set to True if you want to use UV maps
    use_border=False  # TODO: Set to True if you want to use border masks for better training
)

Loaded weights from models/best_model.pth
Found 1700 samples in renders/synthetic_data_pitch_sweep
Train samples: 1360, Val samples: 340


In [4]:
# cut val set in half to create a test set
val_dataset = val_loader.dataset
test_size = len(val_dataset) // 2
test_dataset = torch.utils.data.Subset(val_dataset, range(test_size))
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [5]:
# pick a single batch from test_loader
batch = next(iter(test_loader))
filenames = batch['filename']
gt_uv_t = batch['uv']

ssim_input, ssim_gt_uv = [], []

for i in range(len(filenames)):
    fname = filenames[i]
    
    rgb_raw = np.array(Image.open(os.path.join(DATA_DIR, 'rgb', f'{fname}.jpg'))
                       .convert('RGB').resize((256, 256), Image.BILINEAR))
    gt_img  = np.array(Image.open(os.path.join(DATA_DIR, 'ground_truth', f'{fname}.png'))
                       .convert('RGB').resize((256, 256), Image.BILINEAR))
    gt_uv   = gt_uv_t[i].numpy().transpose(1, 2, 0)
    fg_mask = ~((gt_uv[:,:,0] == gt_uv[:,:,1]) & (gt_uv[:,:,1] == 0))

    # baseline 1: no dewarp (input vs gt)
    rgb_t = torch.from_numpy(rgb_raw).float().permute(2,0,1).unsqueeze(0)
    gt_t  = torch.from_numpy(gt_img).float().permute(2,0,1).unsqueeze(0)
    ssim_input.append(ssim(rgb_t, gt_t, data_range=255).item())

    # baseline 2: GT UV dewarp (upper bound)
    warped_gt = dewarp_with_uv(rgb_raw, gt_uv, out_size=256, mask=fg_mask)
    warped_gt_t = torch.from_numpy(warped_gt).float().permute(2,0,1).unsqueeze(0)
    ssim_gt_uv.append(ssim(warped_gt_t, gt_t, data_range=255).item())

print(f"No dewarp SSIM:   {np.mean(ssim_input):.4f}")
print(f"GT UV SSIM:       {np.mean(ssim_gt_uv):.4f}")
print(f"Your model SSIM:  0.3874")

No dewarp SSIM:   0.1823
GT UV SSIM:       0.4956
Your model SSIM:  0.3874


## debug: compare models directly (256 vs 512)

In [ ]:
# build test-inference directories ensuring they use the same image stems

import os
import shutil
import random

os.makedirs('test-inference', exist_ok=True)
os.makedirs('test-inference-gt', exist_ok=True)

rgb_dir = 'renders/synthetic_data_pitch_sweep/rgb'
gt_dir  = 'renders/synthetic_data_pitch_sweep/ground_truth'

stems = sorted([os.path.splitext(f)[0] for f in os.listdir(rgb_dir) if f.endswith('.jpg')])
selected = random.sample(stems, 20)

for stem in selected:
    shutil.copy(os.path.join(rgb_dir, f'{stem}.jpg'), f'test-inference/{stem}.jpg')
    shutil.copy(os.path.join(gt_dir,  f'{stem}.png'), f'test-inference-gt/{stem}.png')

print(f"Copied {len(selected)} pairs")

Copied 20 pairs


In [17]:
# next run inference.py

In [18]:
import os
import matplotlib.pyplot as plt
from PIL import Image

os.makedirs('comparison', exist_ok=True)

image_files = sorted([
    f for f in os.listdir('test-inference') 
    if os.path.splitext(f)[1].lower() in ['.jpg', '.jpeg', '.png']
])

for i, fname in enumerate(image_files):
    stem = os.path.splitext(fname)[0]
    
    inp  = Image.open(f'test-inference/{fname}')
    r256 = Image.open(f'results-256/rectified_{i}.png')
    r512 = Image.open(f'results-512/rectified_{i}.png')
    gt   = Image.open(f'test-inference-gt/{stem}.png')

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(inp);  axes[0].set_title('Input')
    axes[1].imshow(r256); axes[1].set_title('256')
    axes[2].imshow(r512); axes[2].set_title('512')
    axes[3].imshow(gt);   axes[3].set_title('GT')
    for ax in axes: ax.axis('off')
    plt.suptitle(stem)
    plt.tight_layout()
    plt.savefig(f'comparison/{i:04d}.png', dpi=100, bbox_inches='tight')
    plt.close()

print("Done")

Done


## !!Use the 512 model, the comparison shows that produces better looking results.
Why is SSIM lower then?